# Analysis of GEnome-scale Regulatory and Metabolic (GERM) models

This notebook runs MEWpy's phenotype simulation methods for integrated metabolic-regulatory models, available in **`mewpy.germ.analysis`**:

- **`RFBA`** - Regulatory FBA: the regulatory state is decoded first and switches reactions off before an FBA
- **`SRFBA`** - Steady-state Regulatory FBA: one mixed-integer problem with the metabolic, GPR and regulatory constraints
- **`PROM`** - Probabilistic Regulation of Metabolism: regulator knockouts constrain fluxes by probabilities inferred from expression data
- **`CoRegFlux`** - fluxes constrained by target-gene expression predicted from regulator co-expression

and the related tools: FVA and deletions (`fva`, `single_gene_deletion`, `single_reaction_deletion`), the regulatory truth table, `find_conflicts`, and integrated FVA and deletions (`ifva`, `isingle_*_deletion`). FBA and pFBA run through the model's simulator (`get_simulator(model).simulate(method=...)`).

**Models used** (all in `models/germ/`):
- **E. coli core**: integrated model from [Orth _et al_, 2010](https://doi.org/10.1128/ecosalplus.10.2.1)
- **E. coli iMC1010**: iJR904 GEM + iMC1010 TRN from [Covert _et al_, 2004](https://doi.org/10.1038/nature02456)
- **M. tuberculosis iNJ661**: model and expression data from [Chandrasekaran _et al_, 2010](https://doi.org/10.1073/pnas.1005139107)
- **S. cerevisiae iMM904**: model and data from [Banos _et al_, 2017](https://doi.org/10.1186/s12918-017-0507-0)

**Contents**: the analysis interface and solutions; FBA, pFBA, FVA and deletions; the regulatory truth table; FBA, RFBA and SRFBA on _E. coli_ core (model changes and attached methods, regulator knockouts, COBRApy and REFRAMED models); RFBA, `find_conflicts`, SRFBA and iFVA on iMC1010; PROM on iNJ661; CoRegFlux on iMM904.

In [1]:
# imports
import os
from pathlib import Path

import pandas as pd

from mewpy.io import Engines, Reader, read_model
from mewpy.simulation import SimulationMethod, get_simulator
from mewpy.germ.analysis import (RFBA, SRFBA, PROM, CoRegFlux, fva, ifva, find_conflicts, isingle_regulator_deletion,
                                 predict_gene_expression, regulatory_truth_table, single_gene_deletion,
                                 single_reaction_deletion, slim_coregflux, slim_prom, slim_rfba, slim_srfba,
                                 target_regulator_interaction_probability)

In [2]:
# MEWpy uses CPLEX or Gurobi first when they are installed; this notebook uses SCIP so that
# the results do not depend on the installed solvers
from mewpy.solvers import set_default_solver
set_default_solver('scip')

In [3]:
# readers
path = Path(os.getcwd()).joinpath('models', 'germ')

# E. coli core
core_gem_reader = Reader(Engines.MetabolicSBML, path.joinpath('e_coli_core.xml'))
core_trn_reader = Reader(Engines.BooleanRegulatoryCSV,
                         path.joinpath('e_coli_core_trn.csv'),
                         sep=',',
                         id_col=0,
                         rule_col=2,
                         aliases_cols=[1],
                         header=0)

# E. coli iMC1010
imc1010_gem_reader = Reader(Engines.MetabolicSBML, path.joinpath('iJR904.xml'))
imc1010_trn_reader = Reader(Engines.BooleanRegulatoryCSV,
                            path.joinpath('iMC1010.csv'),
                            sep=',',
                            id_col=0,
                            rule_col=4,
                            aliases_cols=[1, 2, 3],
                            header=0)

# M. tuberculosis iNJ661
inj661_gem_reader = Reader(Engines.MetabolicSBML, path.joinpath('iNJ661.xml'))
inj661_trn_reader = Reader(Engines.TargetRegulatorRegulatoryCSV,
                           path.joinpath('iNJ661_trn.csv'),
                           sep=';',
                           target_col=0,
                           regulator_col=1,
                           header=None)
inj661_gene_expression_path = path.joinpath('iNJ661_gene_expression.csv')

# S. cerevisiae iMM904
imm904_gem_reader = Reader(Engines.MetabolicSBML, path.joinpath('iMM904.xml'))
imm904_trn_reader = Reader(Engines.CoExpressionRegulatoryCSV,
                           path.joinpath('iMM904_trn.csv'),
                           sep=',',
                           target_col=2,
                           co_activating_col=3,
                           co_repressing_col=4,
                           header=0)

## Working with GERM model analysis
The simulation methods `RFBA`, `SRFBA`, `PROM` and `CoRegFlux` share the following attributes:
- `method` - the name of the simulation method
- `model` - the model used to build the problem
- `solver` - the MEWpy solver instance (CPLEX, Gurobi, SCIP or OptLang, the default solver) holding the variables and constraints of the problem
- `synchronized` - whether the problem is up to date with the model

and two methods:
- **`build`** - creates the problem from the model according to the formulation of the method
- **`optimize`** - solves it. It accepts method-specific arguments (initial state, dynamic, etc.) and solver arguments in `solver_kwargs` (`linear`, `minimize`, `constraints`, etc.), which apply to that optimization only.

In [4]:
# reading the E. coli core model
model = read_model(core_gem_reader, core_trn_reader)

# SRFBA(model) only creates the method; build() creates the mixed-integer problem
srfba = SRFBA(model).build()
srfba

Method,SRFBA
Model,e_coli_core
Type,"regulatory, metabolic"
Variables,619
Constraints,951
Objective,Biomass_Ecoli_core: 1.0
Solver,PySCIPOptSolver
Synchronized,True


`optimize` returns a solution with the objective value, the status and the value of every variable. `optimize(to_solver=True)` returns the raw solver solution instead.

In [5]:
solution = srfba.optimize()
solution

SRFBA Solution
 Objective value: 0.8739215067
 Status: Optimal

**`to_frame()`** returns a pandas `DataFrame` with the value of every variable of the solved problem: reaction fluxes, gene and regulator states, and the auxiliary variables created for the regulatory rules (e.g. `rel_(SUCCt2_2>0)`).

In [6]:
solution.to_frame()

,value
ACALD,0.00000
ACALDt,0.00000
ACKr,0.00000
ACONTa,6.00725
ACONTb,6.00725
...,...
rel_(SUCCt2_2>0),0.00000
or_((GLCpts>0)|(PYK>0)|(PFK>0)|(LDH_D>0)|(SUCCt2_2>0)),1.00000
not_(~((GLCpts>0)|(PYK>0)|(PFK>0)|(LDH_D>0)|(SUCCt2_2>0))),0.00000
and_((~((ME2>0)|(ME1>0)))&(~((GLCpts>0)|(PYK>0)|(PFK>0)|(LDH_D>0)|(SUCCt2_2>0)))),0.00000


**`to_summary()`** groups these values:
- `inputs` - exchange reactions with negative flux (uptake)
- `outputs` - exchange reactions with positive flux (secretion)
- `metabolic` - fluxes of the other reactions
- `regulatory` - values of the regulators and targets
- `objective` - the objective value and direction
- `df` - all values, as in `to_frame()`

In [7]:
summary = solution.to_summary()
summary.inputs

,value
EX_glc__D_e,-10.000000
EX_nh4_e,-4.765319
EX_o2_e,-21.799493
EX_pi_e,-3.214895


In [8]:
summary.outputs

,value
EX_co2_e,22.809833
EX_h_e,17.530865
EX_h2o_e,29.175827


In [9]:
summary.metabolic

,value
ACALD,0.000000
ACALDt,0.000000
ACKr,0.000000
ACONTa,6.007250
ACONTb,6.007250
...,...
TALA,1.496984
THD2,0.000000
TKT1,1.496984
TKT2,1.181498


In [10]:
summary.regulatory

,value
Biomass_Ecoli_core,8.739215e-01
FBP,0.000000e+00
GLCpts,1.000000e+01
LDH_D,0.000000e+00
ME1,0.000000e+00
...,...
CRPnoGLC,6.198859e-08
b4124,0.000000e+00
b3868,-6.198859e-08
b4018,1.000000e+00


In [11]:
summary.objective

,value,direction
SRFBA,0.873922,maximize


## FBA and pFBA
**FBA** (Flux Balance Analysis) finds a flux distribution that maximises the objective function subject to the mass balance and flux bound constraints of a GEM model. With the biomass reaction as objective, it predicts the optimal growth rate. For more details consult: [https://doi.org/10.1038/nbt.1614](https://doi.org/10.1038/nbt.1614).

**pFBA** (Parsimonious FBA) keeps the growth rate at 99.9% of its optimum and then minimises the total sum of fluxes, i.e. the most efficient flux distribution [Lewis _et al_, 2010](https://doi.org/10.1038/msb.2010.47). Below, the pFBA growth rate is 0.873 ≈ 0.999 × 0.873922.

`mewpy.germ.analysis` has no FBA or pFBA class: both run through the model's simulator, the same interface used for COBRApy and REFRAMED models.

In [12]:
# the metabolic model only
met_model = read_model(core_gem_reader)
simulator = get_simulator(met_model)

# FBA is the default method
fba_result = simulator.simulate(method=SimulationMethod.FBA)
fba_result

============================================================
Simulation Result
============================================================
Status:              OPTIMAL
Objective (maximize): 0.873922
Method:              FBA
Model:               e_coli_core
Fluxes:              49 non-zero / 95 total
============================================================

In [13]:
# the pFBA objective value is the minimised total flux; the growth rate is the biomass flux
pfba_result = simulator.simulate(method=SimulationMethod.pFBA)
print('total flux:', round(pfba_result.objective_value, 4))
print('growth rate:', round(pfba_result.fluxes['Biomass_Ecoli_core'], 4))

total flux: 517.9224
growth rate: 0.873


## FVA and deletions
**`fva`** computes the minimum and maximum flux of each reaction while the objective is kept at `fraction` of its optimum (default 1.0), and returns a `DataFrame` with the minimum and maximum (columns) of each reaction (index). It can be used to identify the reactions that limit growth.

**`single_reaction_deletion`** and **`single_gene_deletion`** run an FBA for each reaction deletion or gene knockout of the metabolic model. They are faster than iterating through the reactions or genes with `ko()`.

In [14]:
fva(met_model)

,minimum,maximum
ACALD,-1.776357e-15,0.000000e+00
ACALDt,0.000000e+00,-3.197442e-14
ACKr,8.881784e-15,0.000000e+00
ACONTa,6.007250e+00,6.007250e+00
ACONTb,6.007250e+00,6.007250e+00
...,...,...
TALA,1.496984e+00,1.496984e+00
THD2,0.000000e+00,-1.989520e-13
TKT1,1.496984e+00,1.496984e+00
TKT2,1.181498e+00,1.181498e+00


In [15]:
single_reaction_deletion(met_model)

,growth,status
ACALD,0.873922,Optimal
ACALDt,0.873922,Optimal
ACKr,0.873922,Optimal
ACONTa,0.000000,Optimal
ACONTb,0.000000,Optimal
...,...,...
TALA,0.864759,Optimal
THD2,0.873922,Optimal
TKT1,0.864759,Optimal
TKT2,0.866674,Optimal


In [16]:
single_gene_deletion(met_model)

,growth,status
b0351,0.873922,Optimal
b1241,0.873922,Optimal
s0001,0.211141,Optimal
b2296,0.873922,Optimal
b3115,0.873922,Optimal
...,...,...
b2464,0.873922,Optimal
b0008,0.873922,Optimal
b2935,0.873922,Optimal
b2465,0.873922,Optimal


In [17]:
# single gene deletion for specific genes
single_gene_deletion(met_model, genes=met_model.reactions['ACONTa'].genes)

,growth,status
b0118,0.873922,Optimal
b1276,0.873922,Optimal


## Regulatory truth table
**`regulatory_truth_table`** evaluates every regulatory interaction of a model, by default with each regulator at its maximum coefficient. It returns a `DataFrame` with one row per target: `result` is the value of the target and the other columns are the values of the regulators in its rule.

In [18]:
# regulatory truth table of the E. coli core TRN
reg_model = read_model(core_trn_reader)
regulatory_truth_table(reg_model)

,result,surplusFDP,surplusPYR,b0113,b3261,b0400,pi_e,b4401,b1334,b3357,...,TALA,PGI,fru_e,ME2,ME1,GLCpts,PYK,PFK,LDH_D,SUCCt2_2
b0008,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
b0080,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
b0113,0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
b0114,1,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
b0115,1,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CRPnoGLM,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NRI_hi,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NRI_low,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
surplusFDP,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## FBA, RFBA and SRFBA on _E. coli_ core
FBA ignores the TRN. RFBA fixes the regulatory state first and switches off the reactions whose GPR becomes false. SRFBA solves the metabolic, GPR and regulatory constraints together. Both RFBA and SRFBA are described in detail in the iMC1010 sections below.

In [19]:
model = read_model(core_gem_reader, core_trn_reader)
simulator = get_simulator(model)
rfba = RFBA(model).build()
srfba = SRFBA(model).build()

print('FBA:  ', round(simulator.simulate().objective_value, 4))
print('RFBA: ', round(rfba.optimize().objective_value, 4))
print('SRFBA:', round(srfba.optimize().objective_value, 4))

FBA:   0.8739
RFBA:  0.8739
SRFBA: 0.8739


SRFBA reaches the FBA optimum: the solver can choose a regulatory state that does not restrict growth. RFBA's default regulatory state sets each metabolite regulator from the medium (present if its exchange reaction allows uptake) and every other regulator active; the regulators that follow rules are then evaluated until they no longer change. In this medium the regulatory state closes no reaction the FBA optimum uses, so RFBA reaches it too. For example, Nac (`b1988`) represses `b1761`, the gene of glutamate dehydrogenase (GLUDy). Nac follows `NRI_low`, which follows GlnG (`b3868`), active when there is no ammonium. With ammonium in the medium the chain settles to Nac inactive:

In [20]:
for target in ('b1761', 'b1988', 'NRI_low', 'b3868'):
    print(model.get(target).interaction)
print('GLUDy GPR:', model.reactions['GLUDy'].gene_protein_reaction_rule)

state = rfba.settle_regulatory_state(rfba.initial_state())
print({regulator: state[regulator] for regulator in ('nh4_e', 'b3868', 'NRI_low', 'b1988')})

b1761 || 1 = ( ~ (b1988 | (glu__L_e > 0)))
b1988 || 1 = NRI_low
NRI_low || 1 = b3868
b3868 || 1 = ( ~ (nh4_e > 0))
GLUDy GPR: b1761
{'nh4_e': 1000.0, 'b3868': 0.0, 'NRI_low': 0.0, 'b1988': 0.0}


### Model changes and attached methods
A simulation method is initialized with a GERM model. A common workflow is:
1. `model = read_model(reader1, reader2)` - read the model
2. `srfba = SRFBA(model)` - initialize the simulation method
3. `srfba.build()` - build the problem
4. `solution = srfba.optimize()` - perform the optimization
5. `model.reactions['MY_REACTION'].bounds = (0, 0)` - make changes to the model
6. `solution = SRFBA(model).build().optimize()` - initialize, build and optimize again

Here `model` and `srfba` are not connected: optimizing `srfba` again after step 5 gives the result of the unchanged model.

With `attach=True`, the method is attached to the model and rebuilds its problem on the next `optimize` after every change to the model:
1. `model = read_model(reader1, reader2)`
2. `srfba = SRFBA(model, attach=True)`
3. `srfba.build()`
4. `solution = srfba.optimize()`
5. `model.reactions['MY_REACTION'].bounds = (0, 0)`
6. `solution = srfba.optimize()` - the optimization uses the changed model

Changes made inside a `with model:` block are reverted when the block ends. In both examples below, oxygen uptake is closed (anaerobic growth).

In [21]:
# first workflow: build again after the change
model = read_model(core_gem_reader, core_trn_reader)
print('aerobic SRFBA:  ', round(SRFBA(model).build().optimize().objective_value, 4))

with model:
    model.reactions['EX_o2_e'].lower_bound = 0
    print('anaerobic SRFBA:', round(SRFBA(model).build().optimize().objective_value, 4))

aerobic SRFBA:   0.8739
anaerobic SRFBA: 0.2117


In [22]:
# second workflow: the simulator and the attached methods follow the model
model = read_model(core_gem_reader, core_trn_reader)
simulator = get_simulator(model)
rfba = RFBA(model, attach=True).build()
srfba = SRFBA(model, attach=True).build()


def growth_rates():
    return {'FBA': simulator.simulate().objective_value,
            'RFBA': rfba.optimize().objective_value,
            'SRFBA': srfba.optimize().objective_value}


rates = {'aerobic': growth_rates()}
with model:
    model.reactions['EX_o2_e'].lower_bound = 0
    rates['anaerobic'] = growth_rates()
# leaving the with block restored the oxygen uptake
rates['aerobic again'] = growth_rates()
pd.DataFrame(rates).T.round(4)

,FBA,RFBA,SRFBA
aerobic,0.8739,0.8739,0.8739
anaerobic,0.2117,0.2117,0.2117
aerobic again,0.8739,0.8739,0.8739


Without oxygen, RFBA reads the oxygen signal from the medium (`o2_e` is absent because `EX_o2_e` allows no uptake), so ArcA (`b4401`) and Fnr (`b1334`), whose rules are `~(o2_e > 0)`, are active. RFBA reaches the anaerobic FBA optimum, as SRFBA does.

### Regulator knockouts
A regulator knockout never changes FBA, which ignores the TRN. **`isingle_regulator_deletion`** knocks out each regulator with RFBA or SRFBA by setting its state to 0 in `initial_state`; for a regulator that is also a target, the knockout overrides its own rule. The table lists the transcription factors (regulators that are not reactions or metabolites) whose knockout changes RFBA or SRFBA growth.

In [23]:
model = read_model(core_gem_reader, core_trn_reader)
simulator = get_simulator(model)
rfba = RFBA(model).build()
srfba = SRFBA(model).build()

tfs = [reg_id for reg_id, reg in model.regulators.items() if not (reg.is_reaction() or reg.is_metabolite())]
wild_type = pd.Series({'FBA': simulator.simulate().objective_value,
                       'RFBA': rfba.optimize().objective_value,
                       'SRFBA': srfba.optimize().objective_value})
knockouts = pd.DataFrame({
    'FBA': wild_type['FBA'],
    'RFBA': isingle_regulator_deletion(model, regulators=tfs, method='rfba')['growth'],
    'SRFBA': isingle_regulator_deletion(model, regulators=tfs, method='srfba')['growth'],
})
changed = (knockouts - wild_type).abs().gt(1e-6).any(axis=1)
print(len(tfs), 'transcription factors,', changed.sum(), 'knockouts change growth')
pd.concat([wild_type.to_frame('wild type').T, knockouts[changed]]).round(4)

22 transcription factors, 1 knockouts change growth


,FBA,RFBA,SRFBA
wild type,0.8739,0.8739,0.8739
b3261,0.8739,0.0000,0.2117


Only the Fis (`b3261`) knockout changes growth.
- **RFBA** reads the oxygen signal from the medium, so ArcA and Fnr are inactive and pyruvate formate-lyase (PFL) is closed. PdhR is active, as `surplusPYR` is false while its reaction regulators (e.g. GLCpts) start active, so pyruvate dehydrogenase (PDH, rule `~PdhR | Fis`) needs Fis. Without Fis both PDH and PFL are closed, and RFBA predicts no growth.
- **SRFBA** chooses the regulatory state together with the fluxes. The next cell compares its wild-type and knockout solutions.

In [24]:
wt_values = srfba.optimize().values
ko_values = srfba.optimize(initial_state={'b3261': 0}).values

rows = {'Fis (b3261)': 'b3261', 'PdhR (b0113)': 'b0113', 'ArcA (b4401)': 'b4401', 'Fnr (b1334)': 'b1334',
        'Crp (b3357)': 'b3357', 'Nac (b1988)': 'b1988', 'PDH flux': 'PDH', 'NADH16 flux': 'NADH16', 'SUCDi flux': 'SUCDi', 'EX_o2_e flux': 'EX_o2_e',
        'growth': 'Biomass_Ecoli_core'}
pd.DataFrame({'wild type': {name: wt_values[var] for name, var in rows.items()},
              'Fis knockout': {name: ko_values[var] for name, var in rows.items()}}).round(4)

,wild type,Fis knockout
Fis (b3261),1.0000,0.0000
PdhR (b0113),1.0000,1.0000
ArcA (b4401),0.0000,1.0000
Fnr (b1334),0.0000,1.0000
Crp (b3357),0.0000,0.0000
Nac (b1988),-0.0000,0.0000
PDH flux,9.2825,0.0000
NADH16 flux,38.5346,0.0000
SUCDi flux,5.0644,0.0000
EX_o2_e flux,-21.7995,0.0000


In the knockout solution PdhR is active and Fis is off, so the PDH genes (rule `~b0113 | b3261`, i.e. `~PdhR | Fis`) are off. ArcA and Fnr are active and Crp is inactive, which switches off NADH16 and SUCDi (rules of one gene of each below); no oxygen is taken up and growth equals the anaerobic rate computed above.

In [25]:
print('PDH:   ', model.get('b0114').interaction)
print('NADH16:', model.get('b2276').interaction)
print('SUCDi: ', model.get('b0721').interaction)

PDH:    b0114 || 1 = (( ~ b0113) | b3261)
NADH16: b2276 || 1 = ( ~ (b4401 | b1334))
SUCDi:  b0721 || 1 = (( ~ (b4401 | b1334)) | b3357 | b3261)


### COBRApy and REFRAMED models
The same analysis runs on a `RegulatoryExtension`, which adds the TRN to a MEWpy simulator of a COBRApy or REFRAMED model (see the 12-GERM_Models notebook). REFRAMED prefixes its identifiers (`R_`, `M_`, `G_`); the TRN identifiers are renamed to match them, and `id_map` lists the renames. Fis is not a metabolic gene and keeps its identifier `b3261`.

In [26]:
from mewpy.germ.models import RegulatoryExtension

trn_args = dict(sep=',', id_col=0, rule_col=2, aliases_cols=[1], header=0)
gem_file, trn_file = str(path.joinpath('e_coli_core.xml')), str(path.joinpath('e_coli_core_trn.csv'))
models = {
    'read_model': read_model(core_gem_reader, core_trn_reader),
    "RegulatoryExtension, flavor='cobra'": RegulatoryExtension.from_sbml(gem_file, trn_file, flavor='cobra', **trn_args),
    "RegulatoryExtension, flavor='reframed'": RegulatoryExtension.from_sbml(gem_file, trn_file, flavor='reframed', **trn_args),
}

results = {}
for name, integrated in models.items():
    # a RegulatoryExtension holds its simulator
    sim = integrated.simulator if isinstance(integrated, RegulatoryExtension) else get_simulator(integrated)
    results[name] = {
        'FBA': sim.simulate().objective_value,
        'RFBA': RFBA(integrated).build().optimize().objective_value,
        'SRFBA': SRFBA(integrated).build().optimize().objective_value,
        'SRFBA, Fis knockout': SRFBA(integrated).build().optimize(initial_state={'b3261': 0}).objective_value,
        'renamed TRN ids': len(getattr(integrated, 'id_map', {})),
    }
results = pd.DataFrame(results).T
results['renamed TRN ids'] = results['renamed TRN ids'].astype(int)
results.round(4)

,FBA,RFBA,SRFBA,"SRFBA, Fis knockout",renamed TRN ids
read_model,0.8739,0.8739,0.8739,0.2117,0
"RegulatoryExtension, flavor='cobra'",0.8739,0.8739,0.8739,0.2117,0
"RegulatoryExtension, flavor='reframed'",0.8739,0.8739,0.8739,0.2117,160


## RFBA
**`RFBA`** is a phenotype simulation method based on the integration of a GEM model with a TRN at the genome-scale. The TRN consists of a set of regulatory interactions formulated with boolean and propositional logic. The TRN contains a boolean algebra expression for each target gene. This boolean rule determines whether the target gene is active (1) or not (0) according to the state of the regulators (active or inactive). Then, the TRN is integrated with the GEM model using the reactions' GPR rules. It is also common to find metabolites and reactions as regulators/environmental stimuli in the TRN, completing the integration with the GEM model.

In **`RFBA`**, the regulators that are also targets are evaluated first, synchronously and from the initial state, until their states no longer change; a regulator at the end of a chain of regulators gets the state that the chain implies. This gives the regulatory state (regulators' coefficients), which is translated into a metabolic state (metabolic genes' coefficients) by evaluating the regulatory interactions of the metabolic genes. Finally, the reactions whose GPR rules evaluate to false with this metabolic state are switched off, and an FBA is solved.

**`RFBA`** supports steady-state or dynamic phenotype simulations. Dynamic **`RFBA`** performs sequential optimizations: after each one, regulators that are reactions become active if they carry flux and inactive otherwise (metabolite regulators keep their state). It stops when the regulatory state no longer changes.

**`RFBA`** is available in the **`mewpy.germ.analysis`** package. Alternatively, one can use the simple and optimized version **`slim_rfba`**.

For more details consult: [https://doi.org/10.1038/nature02456](https://doi.org/10.1038/nature02456).

This example uses the _E. coli_ iMC1010 model available at _models/germ/iJR904.xml_ and _models/germ/iMC1010.csv_. It consists of the _E. coli_ iJR904 GEM ([Reed _et al_, 2003](https://doi.org/10.1186/gb-2003-4-9-r54)) and the iMC1010 TRN of 1010 regulatory interactions ([Covert _et al_, 2004](https://doi.org/10.1038/nature02456)).

In [27]:
# loading model
model = read_model(imc1010_gem_reader, imc1010_trn_reader)

# objective function
BIOMASS_ID = 'BiomassEcoli'
model.objective = {BIOMASS_ID: 1}
model

============================================================
Model: iJR904
============================================================
Name:                     Reed2003 - Genome-scale metabolic network of Escherichia coli (iJR904)
Types:                    metabolic, regulatory
Reactions:                1083
Metabolites:              768
Genes:                    904
Compartments:             e, c
Exchanges:                150
Interactions:             1010
Targets:                  1010
Regulators:               232
Environmental stimuli:    11
Objective:                BiomassEcoli (maximize)
============================================================

In [28]:
print('FBA:', round(get_simulator(model).simulate().objective_value, 4))
print('RFBA, default initial state:', RFBA(model).build().optimize().objective_value)

FBA: 0.9219
RFBA, default initial state: 0.0


**`RFBA`** starts from an initial regulatory state. Unless given, a metabolite regulator takes the uptake bound of its exchange reaction (|lower bound|, i.e. present if it is in the medium), a reaction regulator its upper bound, and any other regulator its maximum coefficient (active). Setting up a good initial state is difficult: for iMC1010 the default one switches off essential genes and RFBA predicts no growth.

### Find conflicts
**`mewpy.germ.analysis.find_conflicts()`** helps to find the regulators responsible. It decodes the regulatory state from the initial state in two steps (the regulators' rules once, then the targets'), and reports the essential genes (and essential reactions) that this state switches off. Essential genes and reactions are found first with FBA single deletions of the whole model, so this is the slowest step of this notebook.

It returns two DataFrames:
- the essential genes switched off, with their regulatory rule (`interaction`) and the state of each regulator in that rule;
- the essential reactions switched off, with their GPR (`gpr`) and the state of each gene.

It only finds direct conflicts: a regulator in the table may be set by other regulators, metabolites or reactions, which are the ones to change. Run it again with the corrected initial state to check for further conflicts.

In [29]:
repressed_genes, repressed_reactions = find_conflicts(model)
# sorted for a stable display
repressed_genes.sort_index().sort_index(axis=1)

,Stringent,b0676,b4390,interaction
b1092,1.0,NaN,NaN,b1092 || 1 = ( ~ Stringent)
b2574,NaN,NaN,1.0,b2574 || 1 = ( ~ b4390)
b3730,NaN,0.0,NaN,b3730 || 1 = b0676


In [30]:
repressed_reactions.sort_index().sort_index(axis=1)

,b1092,b3730,gpr
G1PACT,NaN,0.0,b3730
MCOATA,0.0,NaN,b1092
UAGDP,NaN,0.0,b3730


Three essential genes are switched off: _b1092_ (rule `~Stringent`), _b2574_ (`~b4390`) and _b3730_ (`b0676`). The second table shows the essential reactions they catalyse (MCOATA needs _b1092_; G1PACT and UAGDP need _b3730_). `Stringent` is set directly, but _b4390_ and _b0676_ are targets with their own rules:

In [31]:
# b4390 is active in high-NAD conditions (an environmental stimulus);
# b0676 is active if neither the acgam_e metabolite is present nor the AGDC reaction is active
print(model.get('b4390').interaction)
print(model.get('b0676').interaction)

# their signals in RFBA's default initial state
default_state = RFBA(model).build().initial_state()
print({regulator: default_state[regulator] for regulator in ['Stringent', 'high-NAD', 'acgam_e', 'AGDC']})

b4390 || 1 = high-NAD
b0676 || 1 = ( ~ ((acgam_e > 0) | (AGDC > 0)))
{'Stringent': 1.0, 'high-NAD': 1.0, 'acgam_e': 0.0, 'AGDC': 999999.0}


By default, acgam_e is absent (its exchange allows no uptake) but the AGDC regulator starts at its upper bound (active), and `Stringent` and `high-NAD` start active. The initial state below therefore sets the stringent response, the high-NAD signal and AGDC inactive. Each of the three is needed: without any one of them RFBA predicts no growth.

In [32]:
# initial state inferred from the find_conflicts results
initial_state = {
    'Stringent': 0.0,
    'high-NAD': 0.0,
    'AGDC': 0.0,
}

# steady-state RFBA
rfba = RFBA(model).build()
solution = rfba.optimize(initial_state=initial_state)
solution

RFBA Solution
 Objective value: 0.9205101727
 Status: Optimal

In [33]:
# without one of the three changes
for regulator in initial_state:
    partial = {key: value for key, value in initial_state.items() if key != regulator}
    print(f'without {regulator}:', rfba.optimize(initial_state=partial).objective_value)

without Stringent: 0.0


without high-NAD: 0.0
without AGDC: 0.0


In [34]:
# using the slim version
slim_rfba(model, initial_state=initial_state)

0.9205101727032998

In [35]:
# dynamic RFBA
dynamic_solution = rfba.optimize(initial_state=initial_state, dynamic=True)
dynamic_solution.solutions

{'t_0': RFBA Solution
  Objective value: 0.9205101727
  Status: Optimal,
 't_1': RFBA Solution
  Objective value: 0.9205101727
  Status: Optimal}

## SRFBA
**`SRFBA`** performs a single steady-state simulation using both the metabolic and the regulatory constraints of the integrated model. It uses Mixed-Integer Linear Programming to solve the nested boolean algebra expressions of the regulatory layer (regulatory interactions) and the metabolic layer (GPR rules). Hence, it adds auxiliary variables for intermediate boolean variables and operators, and a boolean variable and constraint for each reaction that links the GPR to the reaction flux.

**`SRFBA`** only supports steady-state simulations. A target fixed in `initial_state` overrides its regulatory rule.

**`SRFBA`** is available in the **`mewpy.germ.analysis`** package. Alternatively, one can use the simple and optimized version **`slim_srfba`**.

For more details consult: [https://doi.org/10.1038/msb4100141](https://doi.org/10.1038/msb4100141).

This example uses the iMC1010 model of the RFBA section.

**`SRFBA`** needs no initial state: the solver chooses the regulatory state together with the fluxes, to maximise the objective. Regulators that are reactions are linked to their fluxes; a regulator that is a metabolite (e.g. `o2_e`) is active exactly when its exchange reaction carries uptake.

In [36]:
# steady-state SRFBA
srfba = SRFBA(model).build()
solution = srfba.optimize()
solution

SRFBA Solution
 Objective value: 0.8517832812
 Status: Optimal

In [37]:
# using the slim version
slim_srfba(model)

0.851783281176895

SRFBA's growth rate, 0.8517832812, is below RFBA's with the corrected initial state (0.9205101727) and FBA's (0.9219, see the RFBA section). SRFBA links reaction regulators to their fluxes and metabolite regulators to their uptake, while RFBA takes them from the medium and the initial state, so the two methods reach different regulatory states.

## iFVA and iDeletions
**`ifva`** is the integrated version of FVA: it computes the minimum and maximum flux of each reaction with RFBA or SRFBA (`method='rfba'` or `'srfba'`, the default), with the objective kept at `fraction` of its optimum. It returns a `DataFrame` with the minimum and maximum (columns) of each reaction (index).

The `mewpy.germ.analysis` package also includes **`isingle_gene_deletion`**, **`isingle_reaction_deletion`**, and **`isingle_regulator_deletion`** (used above) to inspect _in silico_ genetic strategies in integrated GERM models.

In [38]:
# iFVA of the first fifteen reactions with SRFBA; a fraction below 1 relaxes the objective constraint
reactions_ids = list(model.reactions)[:15]
ifva(model, fraction=0.9, reactions=reactions_ids, method='srfba')

,minimum,maximum
12PPDt,0.000000,0.000000
2DGLCNRx,0.000000,0.000000
2DGLCNRy,0.000000,0.000000
2DGULRx,0.000000,0.000000
2DGULRy,0.000000,0.000000
3HCINNMH,0.000000,0.000000
3HPPPNH,0.000000,0.000000
4HTHRS,0.000000,0.000000
5DGLCNR,-1.364861,0.000000
A5PISO,0.032197,0.035775


## PROM
**`PROM`** is a probabilistic-based phenotype simulation method for integrated models. This method circumvents discrete constraints created by **`RFBA`** and **`SRFBA`**. This method uses a continuous approach: reactions' constraints are proportional to the probabilities of related genes being active. The probability of an active metabolic gene is inferred from the TRN and gene expression dataset. In detail, gene probability is calculated according to the number of samples that the gene is active when its regulator is inactive.

**`PROM`** performs a single steady-state simulation using the probabilistic-based constraints to limit flux through some reactions. This method cannot perform wild-type phenotype simulations though, as probabilities are calculated for single regulator deletion. Hence, this method is adequate to predict the effect of regulator perturbations.

**`PROM`** can generate a **`KOSolution`** containing the solution of each regulator knock-out.

**`PROM`** is available in the **`mewpy.germ.analysis`** package. Alternatively, one can use the simple and optimized version **`slim_prom`**.

For more details consult: [https://doi.org/10.1073/pnas.1005139107](https://doi.org/10.1073/pnas.1005139107).

This example uses the _M. tuberculosis_ iNJ661 model available at _models/germ/iNJ661.xml_, its TRN _models/germ/iNJ661_trn.csv_ (target-regulator pairs), and the gene expression dataset _models/germ/iNJ661_gene_expression.csv_.

In [39]:
# loading model
model = read_model(inj661_gem_reader, inj661_trn_reader)

# objective function
BIOMASS_ID = 'biomass_Mtb_9_60atp_test_NOF'
model.objective = {BIOMASS_ID: 1}

wild_type_growth = get_simulator(model).simulate().objective_value
print('wild-type FBA growth rate:', round(wild_type_growth, 5))
model

wild-type FBA growth rate: 0.0522


============================================================
Model: iNJ661
============================================================
Name:                     M. tuberculosis iNJ661 model - Jamshidi et al 2007
Types:                    metabolic, regulatory
Reactions:                1028
Metabolites:              828
Genes:                    661
Compartments:             c, e
Exchanges:                88
Interactions:             178
Targets:                  178
Regulators:               30
Environmental stimuli:    29
Objective:                biomass_Mtb_9_60atp_test_NOF (maximize)
============================================================

**`PROM`** phenotype simulation requires an initial state that must be inferred from the TRN and gene expression dataset.
Besides, the format of the initial state is slightly different from **`RFBA`** and **`SRFBA`** initial states. **`PROM`**'s initial state must be a dictionary in the following format:
- keys -> tuple of target gene and regulator identifiers
- value -> probability of this regulatory interaction inferred from the gene expression dataset

<br>

**`mewpy.omics`** package contains the required methods to perform a quantile preprocessing of the gene expression dataset. Then, `target_regulator_interaction_probability()` infers **`PROM`**'s initial state.

In [40]:
# the dataset has rows named 'Negative', which are not gene identifiers: drop them
data = pd.read_csv(inj661_gene_expression_path, sep=';', index_col=0, header=None)
print("rows named 'Negative':", (data.index == 'Negative').sum())
data = data[data.index != 'Negative']
print('expression data (genes x samples):', data.shape)

# PROM target-regulator interaction probabilities using the quantile preprocessing pipeline
from mewpy.omics import ExpressionSet

expression = ExpressionSet.from_dataframe(data)
quantile_expression, binary_expression = expression.quantile_pipeline()
initial_state, _ = target_regulator_interaction_probability(model,
                                                            expression=quantile_expression,
                                                            binary_expression=binary_expression)
print(len(initial_state), 'regulator-target probabilities')
pd.Series(initial_state).head()

rows named 'Negative': 25
expression data (genes x samples): (4295, 437)


217 regulator-target probabilities


Rv3396c  Rv0001     0.590361
         Rv3575c    0.510574
         Rv3676     0.647059
Rv3411c  Rv0001     0.542169
         Rv3575c    1.000000
dtype: float64

In [41]:
# PROM knocks out each regulator in turn
prom = PROM(model).build()
solution = prom.optimize(initial_state=initial_state)

prom_growth = pd.DataFrame({ko: {'growth': sol.objective_value, 'status': sol.status.name}
                            for ko, sol in solution.solutions.items()}).T
reduced = prom_growth[prom_growth['growth'] < wild_type_growth - 1e-6]
print(len(prom_growth), 'regulator knockouts,', len(reduced), 'reduce growth below the wild type')
reduced.sort_values('growth')

30 regulator knockouts, 7 reduce growth below the wild type


,growth,status
ko_Rv3575c,0.012514,OPTIMAL
ko_Rv3676,0.014232,OPTIMAL
ko_Rv2069,0.02405,OPTIMAL
ko_Rv1267c,0.025034,OPTIMAL
ko_Rv0001,0.028301,OPTIMAL
ko_Rv2711,0.032241,OPTIMAL
ko_Rv0485,0.042064,OPTIMAL


The other knockouts leave growth at the wild-type value: the probabilities of their targets do not restrict the fluxes needed for growth.

In [42]:
# the slim version performs a single knockout (the first regulator if regulator is None)
slim_prom(model, initial_state=initial_state, regulator='Rv0001')

0.028301046082039726

## CoRegFlux
**`CoRegFlux`** is a linear regression-based phenotype simulation method for integrated models. This method circumvents discrete constraints created by **`RFBA`** and **`SRFBA`**. **`CoRegFlux`** uses a continuous approach: reactions' constraints are proportional (using soft plus activation function) to the predicted expression of related genes. This method uses a linear regression model to predict the expression of a target gene as function of the co-expression of its regulators (co-activators and co-repressors). To train a linear regression model, **`CoRegFlux`** uses the target gene expression and regulators' influence scores* from a training dataset. Then, this model is used to make predictions of the target gene expression in the experiment (test) dataset.

*Influence score is a correlation-based score for the activation or repression of a regulator inferred with CoRegNet available at [https://doi.org/10.1093/bioinformatics/btv305](https://doi.org/10.1093/bioinformatics/btv305).

**`CoRegFlux`** performs a single steady-state simulation using the linear regression model predictions to limit flux through some reactions. Hence, this method can predict the phenotypic behavior of an organism in all environmental conditions available in the gene expression dataset. However, this method must use a different training dataset to infer regulators' influence scores and train the linear regression models. **`CoRegFlux`** can also perform dynamic simulations for a series of time steps. At each time step, dynamic **`CoRegFlux`** updates metabolite concentrations and biomass yield with an Euler step. These values are then translated into additional constraints to be added to the steady-state simulation.

**`CoRegFlux`** can generate a **`Solution`** containing the solution for a single environmental condition in the experiment dataset. In addition, **`CoRegFlux`** can generate a **`DynamicSolution`** containing time-step solutions for a single environmental condition in the experiment dataset.

**`CoRegFlux`** is available in the **`mewpy.germ.analysis`** package. Alternatively, one can use the simple and optimized version **`slim_coregflux`**.

For more details consult: [https://doi.org/10.1186/s12918-017-0507-0](https://doi.org/10.1186/s12918-017-0507-0).

This example uses the following model and data:
- _S. cerevisiae_ iMM904 model available at _models/germ/iMM904.xml_,
- _S. cerevisiae_ TRN inferred with CoRegNet and available at _models/germ/iMM904_trn.csv_,
- _S. cerevisiae_ training gene expression dataset available at _models/germ/iMM904_gene_expression.csv_,
- _S. cerevisiae_ influence scores inferred with CoRegNet in the gene expression dataset available at _models/germ/iMM904_influence.csv_,
- _S. cerevisiae_ experiments gene expression dataset available at _models/germ/iMM904_experiments.csv_.

In [43]:
# loading model
model = read_model(imm904_gem_reader, imm904_trn_reader)

# objective function
BIOMASS_ID = 'BIOMASS_SC5_notrace'
model.objective = {BIOMASS_ID: 1}
print('wild-type FBA growth rate:', round(get_simulator(model).simulate().objective_value, 4))
model

wild-type FBA growth rate: 0.2879


============================================================
Model: iMM904
============================================================
Name:                     S. cerevisae iMM904 model - Mo et al 2009
Types:                    metabolic, regulatory
Reactions:                1577
Metabolites:              1226
Genes:                    905
Compartments:             8 compartments
Exchanges:                164
Interactions:             3748
Targets:                  3748
Regulators:               201
Environmental stimuli:    199
Objective:                BIOMASS_SC5_notrace (maximize)
============================================================

**`CoRegFlux`** phenotype simulation requires an initial state that must be inferred from the TRN, gene expression dataset, influence score matrix and experiments gene expression dataset. This initial state contains the predicted gene expression of target metabolic genes available in the GEM model.
<br>
**`mewpy.germ.analysis.coregflux`** module includes the tools to infer **`CoRegFlux`**'s initial state. These methods create the linear regression models to predict targets' expression according to the experiments gene expression dataset. One just has to load expression, influence and experiments CSV files using `mewpy.omics.ExpressionSet`.

HINT: the `predict_gene_expression` method might be time-consuming for some gene expression datasets. One can save the predictions into a CSV file and then load it afterwards using `mewpy.omics.ExpressionSet.from_csv()`.

In [44]:
from mewpy.omics import ExpressionSet

# HINT: the following line loads pre-computed predictions instead of the remaining lines of this cell
# gene_expression_prediction = ExpressionSet.from_csv(path.joinpath('iMM904_gene_expression_prediction.csv'),
#                                                     sep=',', index_col=0, header=0).dataframe

expression = ExpressionSet.from_csv(path.joinpath('iMM904_gene_expression.csv'), sep=';', index_col=0, header=0).dataframe
influence = ExpressionSet.from_csv(path.joinpath('iMM904_influence.csv'), sep=';', index_col=0, header=0).dataframe
experiments = ExpressionSet.from_csv(path.joinpath('iMM904_experiments.csv'), sep=';', index_col=0, header=0).dataframe

# predicted expression of the target genes (rows) in each experiment (columns)
gene_expression_prediction = predict_gene_expression(model=model, influence=influence, expression=expression,
                                                     experiments=experiments)
gene_expression_prediction

,yB8n055.03II01.batch.p1,yB8n056.03II01.batch.p2,yB8n057.03II01.batch.p3,yB8n058.03II01.batch.p4,yB8n059.03II01.batch.p5,yB8n060.03II01.batch.p6,yB8n061.03II01.batch.p7,yB8n062.03II01.batch.p8,yB8n063.03II01.batch.p9,yB8n064.03II01.batch.p10,yB8n065.03II01.batch.p11,yB8n066.03II01.batch.p12,yB8n044.Low.D.chemostat.vs..High.D.chemostat
YMR056C,10.310906,9.374551,9.738155,9.738831,10.383624,9.973555,9.627858,11.579884,11.163626,11.706841,11.655439,9.197184,10.402504
YBR085W,7.999073,7.978798,7.926310,7.744950,7.655623,7.433795,7.883342,6.681351,6.955076,6.468447,6.930435,7.745075,7.469748
YJR155W,8.719261,8.845556,8.779802,8.848888,8.970303,8.862843,8.714360,9.761043,9.762521,9.905502,9.747369,8.684471,9.510130
YDL243C,6.413120,6.802390,6.658224,6.574777,5.948326,6.198785,6.742563,6.209202,6.547471,5.949884,6.172434,6.455557,6.576763
YHR047C,8.230008,8.768411,8.551748,8.697505,8.882710,9.125730,8.896929,8.868654,8.837522,8.688578,8.392062,9.176441,8.538216
...,...,...,...,...,...,...,...,...,...,...,...,...,...
YML002W,7.757591,7.533461,7.617832,7.536168,7.296171,6.948933,7.461951,7.456734,7.535741,7.428243,7.784473,7.058646,7.606172
YML030W,10.530161,10.486551,10.464589,10.667886,10.675506,10.908625,10.757949,11.978087,11.939201,11.908917,11.600359,10.584607,10.869762
YML053C,10.243145,10.102137,10.054621,10.076379,10.088078,10.059266,10.338693,10.617780,10.593350,10.525567,10.571640,9.983406,10.491541
YML087C,6.589562,6.591232,6.598206,6.712599,6.727422,6.741082,6.576150,7.834831,7.822760,7.842013,7.334910,6.708636,6.898609


In [45]:
# steady-state simulation only requires the initial state of a given experiment (the first experiment in this case)
initial_state = list(gene_expression_prediction.to_dict().values())
co_reg_flux = CoRegFlux(model).build()
solution = co_reg_flux.optimize(initial_state=initial_state[0])
solution

CoRegFlux Solution
 Objective value: 0.2878657037
 Status: Optimal

In [46]:
# using the simple version of CoRegFlux
slim_coregflux(model, initial_state=initial_state[0])

0.2878657037042877

For the first experiment, the predicted expression does not restrict growth: CoRegFlux returns the wild-type FBA growth rate.

A dynamic simulation takes one initial state per time step, plus the initial metabolite concentrations (mM) and biomass (gDW/L). At each step, the uptake bounds follow the concentrations, and the concentrations and biomass are then updated with an Euler step. The first 12 experiments are samples of a batch culture (`batch.p1` to `batch.p12`; the last one is a chemostat comparison), so they are used here as 12 time steps of 1 h each. The file does not give the sampling times, so this time scale is illustrative.

In [47]:
# dynamic simulation: initial metabolite concentrations, biomass and one initial state per time step
batch_samples = [column for column in gene_expression_prediction.columns if '.batch.' in column]
batch_states = [gene_expression_prediction[column].to_dict() for column in batch_samples]
metabolites = {'glc__D_e': 16.6, 'etoh_e': 0}
biomass = 0.45
time_steps = list(range(1, len(batch_samples) + 1))
print(len(batch_samples), 'batch samples; not used:', [c for c in gene_expression_prediction.columns if c not in batch_samples])

co_reg_flux = CoRegFlux(model).build()
solution = co_reg_flux.optimize(initial_state=batch_states,
                                metabolites=metabolites,
                                biomass=biomass,
                                time_steps=time_steps)

pd.DataFrame({step: {'growth': sol.objective_value, 'status': sol.status.name,
                     'EX_glc__D_e': sol.values['EX_glc__D_e'], 'EX_etoh_e': sol.values['EX_etoh_e']}
              for step, sol in solution.solutions.items()}).T

12 batch samples; not used: ['yB8n044.Low.D.chemostat.vs..High.D.chemostat']


,growth,status,EX_glc__D_e,EX_etoh_e
t_1,0.287866,OPTIMAL,-10.0,15.815475
t_2,0.287866,OPTIMAL,-10.0,15.815475
t_3,0.181838,OPTIMAL,-5.537146,8.185478
t_4,0.029163,OPTIMAL,0.0,-4.72844
t_5,0.029163,OPTIMAL,0.0,-4.72844
t_6,0.029163,OPTIMAL,0.0,-4.72844
t_7,0.029163,OPTIMAL,0.0,-1.696736
t_8,0.029163,OPTIMAL,0.0,-2.707304
t_9,0.029163,OPTIMAL,0.0,-1.696736
t_10,0.029163,OPTIMAL,0.0,-4.72844


Glucose is taken up during the first three steps, while ethanol is secreted. Once glucose is exhausted, the cells grow slowly on the ethanol produced (negative `EX_etoh_e`). When little ethanol is left, the uptake it allows can no longer meet the fixed ATP maintenance (`ATPM`), so the last two steps are infeasible.

In [48]:
model.reactions['ATPM'].bounds

(1.0, 1.0)

## Summary

| Method | Needs | Use |
|---|---|---|
| FBA, pFBA | metabolic model | `get_simulator(model).simulate(method=...)` |
| RFBA | integrated model and a suitable initial regulatory state (`find_conflicts` helps to set it) | regulatory state decoded first, then FBA |
| SRFBA | integrated model | regulatory state and fluxes solved together (MILP) |
| PROM | integrated model and expression data | effect of regulator knockouts |
| CoRegFlux | integrated model, influence scores and expression data | condition-specific and dynamic simulations |

Check that the metabolic model grows (FBA) and that the medium is set before running the integrated methods.